# Customer Segmentation

This notebook develops a customer segmentation framework for the Retail Banking Customer Intelligence project.

The analysis uses the account-level analytical dataset constructed in `02_analytical_dataset.ipynb`, where each observation represents one bank account.

The objective is to identify meaningful groups of customers based primarily on their banking behavior, product relationships, transaction activity, cash flows, balances, and transaction patterns.

The workflow includes:

1. Analytical dataset validation
2. Feature selection for segmentation
3. Distribution and outlier analysis
4. Feature transformation
5. Scaling and preprocessing
6. Clustering model comparison
7. Cluster validation
8. Customer segment profiling
9. Business interpretation

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database import engine

with engine.connect():
    print('PostgreSQL connection successful.')

PostgreSQL connection successful.


## 1. Data Loading and Validation

The segmentation analysis uses the account-level analytical dataset developed in the previous stage of the project.

Before preprocessing or clustering, the analytical population and candidate features are validated to ensure that the segmentation is based on a consistent one-row-per-account structure.

In [3]:
customer_360 = pd.read_parquet(
    PROJECT_ROOT / 'data' / 'processed' / 'customer_360.parquet'
)

In [4]:
customer_360.head(5)

,account_id,account_open_date,frequency,owner_client_id,birth_date,gender,age,account_district_id,client_district_id,district_mismatch,district_name,region,inhabitants,urban_ratio,average_salary,unemployment_rate_96,entrepreneurs_per_1000,crimes_96,has_disponent,card_type,has_card,n_standing_orders,total_standing_order_amount,household_payment_amount,insurance_payment_amount,leasing_payment_amount,loan_payment_amount,uncategorized_order_amount,loan_amount,loan_duration,loan_payment,has_loan,n_transactions,first_transaction_date,last_transaction_date,active_days,observation_days,transactions_per_month,active_day_ratio,n_inflows,monthly_inflow,monthly_outflow,avg_inflow_amount,avg_outflow_amount,avg_balance,min_balance,max_balance,balance_std,has_negative_balance,negative_balance_ratio,cash_deposit_ratio,cash_withdrawal_ratio,card_withdrawal_ratio,transfer_in_ratio,transfer_out_ratio
0,1,1995-03-24,MONTHLY,1,1970-12-13,Female,28,18,18,False,Pisek,south Bohemia,70699,65.30,"8,968.00",3.35,131.00,1910,False,NaN,False,1,"2,452.00","2,452.00",0.00,0.00,0.00,0.00,NaN,NaN,NaN,False,239,1995-03-24,1998-12-31,187,1379,5.28,0.14,102,"4,289.42","3,992.17","1,905.10","1,320.10","15,994.01","1,000.00","30,414.80","3,656.89",False,0.00,0.05,0.41,0.00,0.19,0.17
1,2,1993-02-26,MONTHLY,2,1945-02-04,Male,53,1,1,False,Hl.m. Praha,Prague,1204953,100.00,"12,541.00",0.43,167.00,99107,True,NaN,False,2,"10,638.70","7,266.00",0.00,0.00,"3,372.70",0.00,"80,952.00",24.00,"3,373.00",True,478,1993-02-26,1998-12-31,369,2135,6.82,0.17,152,"22,770.17","22,162.40","10,506.93","4,768.18","36,540.70","1,100.00","69,302.00","11,028.14",False,0.00,0.03,0.50,0.00,0.15,0.19
2,3,1997-07-07,MONTHLY,4,1956-12-01,Male,42,5,5,False,Kolin,central Bohemia,95616,51.40,"9,307.00",4.43,118.00,3040,True,NaN,False,3,"5,001.00","1,135.00","3,539.00",0.00,0.00,327.00,NaN,NaN,NaN,False,117,1997-07-07,1998-12-31,100,543,6.56,0.18,46,"9,701.50","6,837.10","3,762.15","1,717.79","26,973.51","1,000.00","53,446.50","10,508.66",False,0.00,0.24,0.27,0.00,0.00,0.33
3,4,1996-02-21,MONTHLY,6,1919-09-22,Male,79,12,12,False,Pribram,central Bohemia,107870,58.00,"8,754.00",4.31,137.00,3868,False,NaN,False,2,"3,363.00","3,363.00",0.00,0.00,0.00,0.00,NaN,NaN,NaN,False,186,1996-02-21,1998-12-31,151,1045,5.42,0.14,65,"5,602.88","4,620.63","2,959.17","1,310.95","21,753.45",800.00,"34,870.00","5,009.98",False,0.00,0.01,0.33,0.00,0.18,0.32
4,5,1997-05-30,MONTHLY,7,1929-01-25,Male,69,15,15,False,Cesky Krumlov,south Bohemia,58796,51.90,"9,045.00",3.60,124.00,1879,False,NaN,False,1,"2,668.00","2,668.00",0.00,0.00,0.00,0.00,NaN,NaN,NaN,False,84,1997-05-30,1998-12-31,69,581,4.40,0.12,35,"5,107.48","3,635.85","2,785.29","1,416.25","25,090.07",600.00,"32,036.20","4,752.47",False,0.00,0.01,0.42,0.00,0.23,0.17
